WebSocket 연결 및 인증

학습 목표

이 장을 통해 다음을 학습할 수 있습니다.

WebSocket 프로토콜의 기본 개념과 특징을 이해합니다.
Websocket 프로토콜 기초

ASGI
Djago Channels 구조

djago Channels는 Djago에 비동기 및 Websocket 지원을 추가하는 확장입니다.

ASGI (Asynchroouse Server Gateway Interface)
Channels는 aSGI를 사용하여 비동기 요청을 처리합니다. ASGI는 WSGI의 비동기 버전으로, Websocket고ㅜㅏ 같은 장기 연결을 지원합니다. 

async def application(scope, recieve, send):

Consumer 패턴
Ch\annels는 Consumer 패턴을 사용하여 WebSocket 연결을 처리합니다. Consumer는 Websocket 이벤트를 처리하느 클래스입니다.

connect:
disconnect:
receicve

JWT JSON Web Toek n구조 

Header: 토큰 타입과 알고리즘
Paylaod: 
Signature: 서명

위조 방지
Websocket 인증은 연결 직전의 HTTP Upgrade 초기 핸드셰이크 

초기 연결 때만 헤더/쿠키가 전달된다
연결이 성립된 이후 헤더가 ㅇ벗어서 추가 헤더 기반 인증이 불가능하다
Websocket API는 사용자 정의 헤더 설정이 제한적이라 실무에서는 브라우저가 자동 포함하는 쿠키로 토큰을 전달하는 방식이 흔다하다.

인증 미들웨어 구현


RealtimeSTTConsumer 구조

realtime_stt_consumer.py의 Consumer는 Handler 패턴을 사용하여 책임을 분리합니다.


In [ ]:
import asyncio
from typing import Optional


class AsyncWebSocketConsumer:
    pass

class SessionContext:
    pass


class RealtimeSTTConsumer(AsyncWebSocketConsumer):
    """실시간 STT WebSocket Consumer
    오디오 데이터를 실시간으로 수신하고 파일 시스템에 저장합니다.
    """

    async def connect(self) -> None:
        await self.accept()

        # 세션 컨텍스트 초기화
        self.session_context: Optional[SessionContext] = None
        self.stt_processing_task: asyncio.Task[None] | None = None

        # Handler 초기화
        self.message_handler = MessageHandler(self)
        self.response_handler = ResponseHandler()
        self.stt_handler = STTHandler()

    
    async def disconnect(self, close_code: int) -> None:
        """WebSocket 연결 종료"""
        if self.stt_processing_task:
            # cancel 로직은 실행되고 있는 task를 취소한다
            self.stt_processing_task.cancel()
        
        await self.message_handler.close_queue()

        # 최종 메타데이터 저장
        # session이 초기화된 상태라면 None이 아닐것이라서
        if self.session_context:
            # SessionContext에서 필요한 정보를 추출하여 메타데이터를 저장합니다.
            # 예: self.session_context.save_metadata()와 같은 메서드 호출로 명확하게 처리
            if hasattr(self.session_context, "save_metadata"):
                self.session_context.save_metadata()
            
    async def receive(self, text_data: str | None = None, bytes_data: bytes | None = None) -> None:
        """메시지 수신 처리"""
        if text_data is not None:
            result = await self.message_handler.handle_text(text_data)
            if result is not None:
                self.session_context = result

        elif bytes_data:
            if self.session_context:
                self.session_context = await self.message_handler.handle_audio(...)
            

    # Handler voxjs
    # 

        


Handler 패턴

Consumer는 각 책임을 Handler 클래스로 분리합니다.
MessageHandler: Websocket 메시지 처리
ResponseHandler: 응답 전송
STTHandler: STT 스트림 처리
AudioHandler: 오디오 청크 생성

단일 책임 원칙
테스트 용이성
재사용성
유지보수성

연결 수립 처리

connect 메서드에서 다음을 수행합니다.

1. 연결 수락 
2. Handler 초기화
3. 세션 컨텍스트 초기화

In [ ]:
async def connect(self) -> None:
    scope = self.scope
    clinet_host = (
        scope.get("client")
    )

    await self.accept()

    self.session_context: SessionContext
    self.stt_processing_task: asyncio.Task[None]

    # 
    self.message_handler = MessageHandler(self)
    self.response_handler = ResponseHandler(self)
    self.stt_handler = STTHandler()

    # 사용자 정보 확인

    # scope 객체? 
     

연결 종료 처리

disconnect 메서드에서 다음을 수행합니다.

STT 처리 테스크 종료: 실행 중인 STT 처리 테스크 취소
오디오 큐 정리: MessageHandler를 통해 종료 신호 전송
메타데이터 저장: SessionContext에서 정보를 추출하여 최종 메타데이터 저장
